# Walk-forward PCA signals

Thin notebook: load canonical prices, compute returns, and run the shared rolling-fit pipeline.

In [ ]:
from sarb.config import load_market_config, load_strategy_config
from sarb.data.canonical import load_canonical_panel
from sarb.features.returns import compute_simple_returns
from sarb.walkforward.rolling_fit import RollingFitPipeline

market = load_market_config("usa")
strategy = load_strategy_config()
panel = load_canonical_panel(
    market.canonical_path,
    market=market.market,
    frequency=market.frequency,
)
returns = compute_simple_returns(panel.prices.dropna(how="all"))
pipeline = RollingFitPipeline(
    pca_components=strategy.settings["pca"]["n_components"],
    estimation_window=strategy.settings["residuals"]["estimation_window"],
    signal_shift=strategy.settings["signals"]["shift_bars"],
    dt_per_year=strategy.settings["ou"]["dt_per_year"],
    thresholds=strategy.settings["signals"],
)
window = next(pipeline.iter_windows(returns))
pipeline.fit_window(window, asset=returns.columns[0], full_returns=returns)